In [32]:
#importing Library - pandas and assigning into a New Variable - pd
import pandas as pd
import numpy as np
import gc
gc.collect()

0

## 1. Understand and Load the Dataset

In [33]:
dataset=pd.read_csv("btcusd_1-min_data.csv")
dataset.head()

,Timestamp,Open,High,Low,Close,Volume
0,1325412060,4.58,4.58,4.58,4.58,0.0
1,1325412120,4.58,4.58,4.58,4.58,0.0
2,1325412180,4.58,4.58,4.58,4.58,0.0
3,1325412240,4.58,4.58,4.58,4.58,0.0
4,1325412300,4.58,4.58,4.58,4.58,0.0


In [34]:
total_rows = dataset.shape[0]
total_columns = dataset.shape[1]

print(f"The dataset has {total_rows} rows and {total_columns} columns.")

The dataset has 1048575 rows and 6 columns.


In [35]:
print(dataset.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 6 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   Timestamp  1048575 non-null  int64  
 1   Open       1048575 non-null  float64
 2   High       1048575 non-null  float64
 3   Low        1048575 non-null  float64
 4   Close      1048575 non-null  float64
 5   Volume     1048575 non-null  float64
dtypes: float64(5), int64(1)
memory usage: 48.0 MB
None


## 2. Data Preprocessing

In [36]:
# Convert Timestamp to Datetime (if applicable)
dataset["Timestamp"] = pd.to_datetime(dataset["Timestamp"], unit="s")
dataset.head()

,Timestamp,Open,High,Low,Close,Volume
0,2012-01-01 10:01:00,4.58,4.58,4.58,4.58,0.0
1,2012-01-01 10:02:00,4.58,4.58,4.58,4.58,0.0
2,2012-01-01 10:03:00,4.58,4.58,4.58,4.58,0.0
3,2012-01-01 10:04:00,4.58,4.58,4.58,4.58,0.0
4,2012-01-01 10:05:00,4.58,4.58,4.58,4.58,0.0


In [37]:
"""
Why do this?
This is the typical starting point for working with time series or date-time data in pandas.
You can now convert, split, or analyze these timestamps easily using pandas’ powerful datetime features.
"""

dataset["Timestamp"] = pd.to_datetime(dataset["Timestamp"], format="%Y-%m-%d %H:%M:%S")

In [38]:

# Create new columns by splitting the timestamp
dataset["Year"]   = dataset["Timestamp"].dt.year
dataset["Month"]  = dataset["Timestamp"].dt.month
dataset["Day"]    = dataset["Timestamp"].dt.day
dataset["Hour"]   = dataset["Timestamp"].dt.hour
dataset["Minute"] = dataset["Timestamp"].dt.minute

dataset.head(3)

,Timestamp,Open,High,Low,Close,Volume,Year,Month,Day,Hour,Minute
0,2012-01-01 10:01:00,4.58,4.58,4.58,4.58,0.0,2012,1,1,10,1
1,2012-01-01 10:02:00,4.58,4.58,4.58,4.58,0.0,2012,1,1,10,2
2,2012-01-01 10:03:00,4.58,4.58,4.58,4.58,0.0,2012,1,1,10,3


In [39]:
dataset = dataset.drop("Timestamp", axis=1)
dataset.head(3)

,Open,High,Low,Close,Volume,Year,Month,Day,Hour,Minute
0,4.58,4.58,4.58,4.58,0.0,2012,1,1,10,1
1,4.58,4.58,4.58,4.58,0.0,2012,1,1,10,2
2,4.58,4.58,4.58,4.58,0.0,2012,1,1,10,3


In [40]:
def get_prediction(row):
    if row['Close'] > row['Open']:
        return "Positive"
    elif row['Close'] < row['Open']:
        return "Negative"
    else:
        return "Neutral"

In [41]:
def generate_predictions(dataset, func):
    for idx, row in dataset.iterrows():
        yield func(row)

In [42]:
# Usage: Build the new column as a generator (never stored all at once)
dataset["Price_Prediction"] = list(generate_predictions(dataset, get_prediction))

In [43]:
int_columns = ['Year','Month','Day','Hour','Minute','Volume']
dataset[int_columns].astype(int)

float_columns = ['Open','High','Low','Close']
dataset[float_columns].astype(float)

dataset.head()

,Open,High,Low,Close,Volume,Year,Month,Day,Hour,Minute,Price_Prediction
0,4.58,4.58,4.58,4.58,0.0,2012,1,1,10,1,Neutral
1,4.58,4.58,4.58,4.58,0.0,2012,1,1,10,2,Neutral
2,4.58,4.58,4.58,4.58,0.0,2012,1,1,10,3,Neutral
3,4.58,4.58,4.58,4.58,0.0,2012,1,1,10,4,Neutral
4,4.58,4.58,4.58,4.58,0.0,2012,1,1,10,5,Neutral


In [44]:
required_order = ['Year','Month','Day','Hour','Minute','Volume','Open','High','Low','Close','Price_Prediction']
dataset = dataset[required_order]
dataset.head()

,Year,Month,Day,Hour,Minute,Volume,Open,High,Low,Close,Price_Prediction
0,2012,1,1,10,1,0.0,4.58,4.58,4.58,4.58,Neutral
1,2012,1,1,10,2,0.0,4.58,4.58,4.58,4.58,Neutral
2,2012,1,1,10,3,0.0,4.58,4.58,4.58,4.58,Neutral
3,2012,1,1,10,4,0.0,4.58,4.58,4.58,4.58,Neutral
4,2012,1,1,10,5,0.0,4.58,4.58,4.58,4.58,Neutral


In [45]:
""" 
For Regeression - Existing Four Scenarios for this Dataset to Replace the NaN/Null values in Any Column :

1.Delete Entire Row(s) those having NaN/Null values in Column
2.Create a Model by considering this Dataset as Semi-supervised Learning, so that We can Predict Values 
for NaN/Null values in Column
3.Replace the NaN/Null values in Column with Central Tendency by Taking Mean/Median/Mode
4.Replace the NaN/Null values in Column with respective to Problem Statement (Requirement/Need) 

For Categorical Data - Existing Scenario for this Dataset to Replace the NaN/Null values in Any Column :

1.Replace the NaN/Null values in Column with Central Tendency by Taking Mode """

' \nFor Regeression - Existing Four Scenarios for this Dataset to Replace the NaN/Null values in Any Column :\n\n1.Delete Entire Row(s) those having NaN/Null values in Column\n2.Create a Model by considering this Dataset as Semi-supervised Learning, so that We can Predict Values \nfor NaN/Null values in Column\n3.Replace the NaN/Null values in Column with Central Tendency by Taking Mean/Median/Mode\n4.Replace the NaN/Null values in Column with respective to Problem Statement (Requirement/Need) \n\nFor Categorical Data - Existing Scenario for this Dataset to Replace the NaN/Null values in Any Column :\n\n1.Replace the NaN/Null values in Column with Central Tendency by Taking Mode '

In [46]:
#Purpose : To Automate the Separation of Both Quantitive and Qualitative Columns with Return Statements of Created List(s) - quan, qual

def quanQual(dataset):
        quan = []
        qual = []

        for columnName in dataset.columns:
            if dataset[columnName].dtype == 'O':
                qual.append(columnName)
                  
            elif dataset[columnName].dtype == 'bool':  # Boolean type
                qual.append(columnName)
            
            elif 'datetime' in str(dataset[columnName].dtype):  # Datetime types
                qual.append(columnName)
                
            elif dataset[columnName].dtype in ['int32', 'float32','int64', 'float64']:  # Numeric types
                quan.append(columnName)
            
            else:
                qual.append(columnName)
        
        return quan, qual

In [47]:
# Assigning the Output of Function - quanQual into Two New Variables - quan and qual, Since the Function returns Two Outputs
quan, qual = quanQual(dataset)

In [48]:
dataset[quan].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 10 columns):
 #   Column  Non-Null Count    Dtype  
---  ------  --------------    -----  
 0   Year    1048575 non-null  int32  
 1   Month   1048575 non-null  int32  
 2   Day     1048575 non-null  int32  
 3   Hour    1048575 non-null  int32  
 4   Minute  1048575 non-null  int32  
 5   Volume  1048575 non-null  float64
 6   Open    1048575 non-null  float64
 7   High    1048575 non-null  float64
 8   Low     1048575 non-null  float64
 9   Close   1048575 non-null  float64
dtypes: float64(5), int32(5)
memory usage: 60.0 MB


In [49]:
dataset[qual].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 1 columns):
 #   Column            Non-Null Count    Dtype 
---  ------            --------------    ----- 
 0   Price_Prediction  1048575 non-null  object
dtypes: object(1)
memory usage: 8.0+ MB


In [50]:
#Calling the Required Varible (quan)
dataset[quan].head(3)

,Year,Month,Day,Hour,Minute,Volume,Open,High,Low,Close
0,2012,1,1,10,1,0.0,4.58,4.58,4.58,4.58
1,2012,1,1,10,2,0.0,4.58,4.58,4.58,4.58
2,2012,1,1,10,3,0.0,4.58,4.58,4.58,4.58


In [51]:
#Checking the Column Name(s) in Another Varible (qual)
dataset[qual].head(3)

,Price_Prediction
0,Neutral
1,Neutral
2,Neutral


In [52]:
#Saving the Preprocessed Dataset as .csv File

dataset.to_csv("Preprocessed_Data_Cryptos.csv",index=False)